# Descrição do Dataset: [Spaceship Titanic](https://www.kaggle.com/competitions/spaceship-titanic/data)

O dataset utilizado nesta competição é uma versão derivativa e sintética que simula um cenário de ficção científica: o objetivo é prever se um passageiro foi transportado para uma dimensão alternativa após a colisão da nave ***Spaceship Titanic*** com uma anomalia espaço-temporal.

Para construir as previsões, o conjunto de dados fornece registros pessoais recuperados do sistema de computadores danificado da nave.

## Arquivos do Projeto

* **`train.csv`**: Registros pessoais de aproximadamente dois terços (~8.700) dos passageiros, utilizados como dados de treino. Contém a variável alvo (`Transported`).
* **`test.csv`**: Registros pessoais do terço restante (~4.300) dos passageiros, utilizados como dados de teste. O objetivo é prever o valor de `Transported` para este conjunto.

---

## Dicionário de Dados (Variáveis)

O conjunto de dados possui as seguintes colunas informativas:

### Identificação e Perfil do Passageiro

* **`PassengerId`**: Um identificador único para cada passageiro no formato `gggg_pp`, onde:
* `gggg` indica o grupo com o qual o passageiro está viajando (frequentemente membros da mesma família).
* `pp` é o número do passageiro dentro daquele grupo.

* **`Name`**: O nome e sobrenome do passageiro.
* **`Age`**: A idade do passageiro.

### Detalhes da Viagem

* **`HomePlanet`**: O planeta de origem do passageiro.
* **`Destination`**: O planeta de destino onde o passageiro iria desembarcar.
* **`Cabin`**: O número da cabine onde o passageiro estava hospedado. Segue o formato `deck/num/side` (conves/número/lado), onde o lado pode ser **P** (*Port* - Bombordo) ou **S** (*Starboard* - Estibordo).
* **`CryoSleep`**: Indica se o passageiro optou por ser colocado em animação suspensa (criossono) durante a viagem. Passageiros em criossono ficam confinados em suas cabines.
* **`VIP`**: Indica se o passageiro pagou por serviços VIP especiais durante a viagem.

### Despesas a Bordo

Valor total que o passageiro gastou em cada uma das muitas comodidades de luxo da Spaceship Titanic:

* **`RoomService`**: Serviço de quarto.
* **`FoodCourt`**: Praça de alimentação.
* **`ShoppingMall`**: Shopping center.
* **`Spa`**: Centro de relaxamento/Spa.
* **`VRDeck`**: Deck de Realidade Virtual.

### Variável Alvo (*Target*)

* **`Transported`**: Indica se o passageiro foi transportado para outra dimensão (**True** ou **False**). Esta é a variável que o modelo deve prever.

# *Setup* do ambiente

Nessa seção iremos:
- Baixar o dataset;
- Baixar as bibliotecas necessárias no projeto.

In [4]:
playground_series_s4e11_path = kagglehub.competition_download('spaceship-titanic')

train = pd.read_csv(playground_series_s4e11_path + "/train.csv")
test = pd.read_csv(playground_series_s4e11_path + "/test.csv")

train.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


# Análise Exploratória dos Dados (EDA)

A intenção dessa seção é ter uma vista geral do *dataset*, a fim de facilitar a preparação de dados para classificação e posterior avaliação do modelo construído.

In [8]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(7)
memory usage: 891.5+ KB


In [6]:
# Descrição geral dos atributos numéricos
train.describe()

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
count,8514.000000,8512.000000,8510.000000,8485.000000,8510.000000,8505.000000
mean,28.827930,224.687617,458.077203,173.729169,311.138778,304.854791
std,14.489021,666.717663,1611.489240,604.696458,1136.705535,1145.717189
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,19.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,27.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,38.000000,47.000000,76.000000,27.000000,59.000000,46.000000
max,79.000000,14327.000000,29813.000000,23492.000000,22408.000000,24133.000000


In [7]:
# Mostra a porcentagem de zeros em cada coluna de gastos
(train[['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']] == 0).mean() * 100

RoomService     64.155067
FoodCourt       62.763143
ShoppingMall    64.270102
Spa             61.244680
VRDeck          63.211780
dtype: float64

# Pré-processamento